In [ ]:
# ???????????
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import f1_score

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout

In [ ]:
# ?????????????? ??? ????????? ?????????? ????????? ????? / Setting all known random seeds
my_code = "Pyslar"
seed_limit = 2 ** 32
my_seed = int.from_bytes(my_code.encode(), "little") % seed_limit

os.environ['PYTHONHASHSEED']=str(my_seed)

random.seed(my_seed)

np.random.seed(my_seed)

tf.compat.v1.set_random_seed(my_seed)

session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
tf.compat.v1.keras.backend.set_session(sess)

In [ ]:
# ?????? ?????? ?? ?????
train_data = pd.read_csv("../datasets/iris_train.csv")

In [ ]:
train_data.head()

In [ ]:
# ????????? ?????? ????????????? ???????
val_size = round(0.2*len(train_data))
print(val_size)

In [ ]:
# ???????? ????????? ? ????????????? ???????
random_state = my_seed
train, val = train_test_split(train_data, test_size=val_size, random_state=random_state)
print(len(train), len(val))

In [ ]:
# ???????? ? ???????? ???????? ??????????? ? ??????? [0,1].
# ??? ????????? ???????????? ?????????? ?????? ????????? ???????.
num_columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
ord_columns = ['species']

ct = ColumnTransformer(transformers=[('numerical', MinMaxScaler(), num_columns)], remainder='passthrough')

ct.fit(train)

In [ ]:
# ??????????? ????????, ??? ?????? ???????? ? DataFrame
sc_train = pd.DataFrame(ct.transform(train))
sc_val = pd.DataFrame(ct.transform(val))

In [ ]:
# ????????????? ???????? ????????
column_names = num_columns + ord_columns
sc_train.columns = column_names
sc_val.columns = column_names

In [ ]:
sc_train

In [ ]:
# ??????? ??????????? ?????????
x_train = sc_train[num_columns]
x_val = sc_val[num_columns]

y_train = (sc_train[ord_columns].values).flatten()
y_val = (sc_val[ord_columns].values).flatten()

In [ ]:
# ???????? ??????? ?????? ????????????? k-?????????
model = KNeighborsClassifier()

In [ ]:
# ?????? ??????
model.fit(x_train, y_train)

In [ ]:
# ???????? ?????? ????????? ????????? ???? ?? ????????????? ???????
pred_val = model.predict(x_val)
f1 = f1_score(y_val, pred_val, average='weighted')
print(f1)

In [ ]:
test = pd.read_csv("../datasets/iris_test.csv")

In [ ]:
test['species'] = ''

In [ ]:
test.head()

In [ ]:
sc_test = pd.DataFrame(ct.transform(test))
sc_test.columns = column_names

In [ ]:
x_test = sc_test[num_columns]

In [ ]:
test['species'] = model.predict(x_test)

In [ ]:
test.head()

In [ ]:
test.to_csv('pyslar.csv', index=False)